In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image, ImageDraw
from IPython.display import display
import cv2
import requests
from io import BytesIO

In [ ]:
!nvidia-smi

**Using Stable Diffusion with Hugging Face**

In [ ]:
# Load pre-trained Stable Diffusion
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# Move to GPU if available
if torch.cuda.is_available():
    pipe = pipe.to("cuda")

In [ ]:
prompt = "a beautiful landscape with mountains and a lake, sunset, highly detailed"
negative_prompt = "blurry, low quality, distorted"

In [ ]:
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,  # More steps = better quality, slower
    guidance_scale=7.5,  # How closely to follow prompt
    height=512,
    width=512
).images[0]

In [ ]:
display(image)

**Advanced Stable Diffusion Features**

1. Image-to-Image:

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

In [ ]:
# Load img2img pipeline
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
img_url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

# Load image from URL
response = requests.get(img_url, timeout=20)
response.raise_for_status()

init_image = Image.open(BytesIO(response.content)).convert("RGB")
init_image = init_image.resize((512, 512))

In [ ]:
image = pipe(
    prompt="transform this into a oil painting style",
    image=init_image,
    strength=0.60,  # How much to transform (0-1)
    num_inference_steps=50
).images[0]

2. Inpainting:

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

In [ ]:
# Load inpainting pipeline
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
image_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
mask_url  = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo_mask.png"

# Load image
img_resp = requests.get(image_url, timeout=20)
image = Image.open(BytesIO(img_resp.content)).convert("RGB").resize((512, 512))

# Load mask (white = edit, black = keep)
mask_resp = requests.get(mask_url, timeout=20)
mask = Image.open(BytesIO(mask_resp.content)).convert("RGB").resize((512, 512))

In [ ]:
result = pipe(
    prompt="a beautiful flower",
    image=image,
    mask_image=mask,
    num_inference_steps=50
).images[0]

result.show()

In [ ]:
result

3. ControlNet (Conditional Control):

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
import cv2

In [ ]:
# Load ControlNet
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
import numpy as np

img_url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

# Load image from URL
response = requests.get(img_url, timeout=20)
response.raise_for_status()

image = np.array(response)
canny_image = cv2.Canny(image, 100, 200)

In [ ]:
output = pipe(
    prompt="a beautiful landscape",
    image=canny_image,
    num_inference_steps=20
).images[0]

**Variational Autoencoders (VAEs)**

In [ ]:
from diffusers import AutoencoderKL

In [ ]:
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
# Encode image to latent
with torch.no_grad():
    latent = vae.encode(image_tensor).latent_dist.sample()

# Decode latent to image
with torch.no_grad():
    image = vae.decode(latent).sample